Copyright (c) Microsoft Corporation. All rights reserved.

Licensed under the MIT License.

![Impressions](https://PixelServer20190423114238.azurewebsites.net/api/impressions/MachineLearningNotebooks/how-to-use-azureml/deployment/deploy-to-cloud/model-register-and-deploy.png)

# Register Model and deploy as Webservice

This example shows how to deploy a Webservice in step-by-step fashion:

 1. Register Model
 2. Deploy Model as Webservice

## Prerequisites
If you are using an Azure Machine Learning Notebook VM, you are all set. Otherwise, make sure you go through the [configuration](../../../configuration.ipynb) Notebook first if you haven't.

In [1]:
# Check core SDK version number
import azureml.core

print("SDK version:", azureml.core.VERSION)

Failure while loading azureml_run_type_providers. Failed to load entrypoint hyperdrive = azureml.train.hyperdrive:HyperDriveRun._from_run_dto with exception (azureml-core 1.0.65 (/home/cody/miniconda3/envs/aml/lib/python3.7/site-packages), Requirement.parse('azureml-core==0.1.0.5621271.*')).
Failure while loading azureml_run_type_providers. Failed to load entrypoint azureml.PipelineRun = azureml.pipeline.core.run:PipelineRun._from_dto with exception (azureml-core 1.0.65 (/home/cody/miniconda3/envs/aml/lib/python3.7/site-packages), Requirement.parse('azureml-core==0.1.0.5621271.*')).
Failure while loading azureml_run_type_providers. Failed to load entrypoint azureml.ReusedStepRun = azureml.pipeline.core.run:StepRun._from_reused_dto with exception (azureml-core 1.0.65 (/home/cody/miniconda3/envs/aml/lib/python3.7/site-packages), Requirement.parse('azureml-core==0.1.0.5621271.*')).
Failure while loading azureml_run_type_providers. Failed to load entrypoint azureml.StepRun = azureml.pipeli

SDK version: 1.0.65


## Initialize Workspace

Initialize a workspace object from persisted configuration.

In [2]:
from azureml.core import Workspace

ws = Workspace.from_config()
print(ws.name, ws.resource_group, ws.location, ws.subscription_id, sep = '\n')

Azure-ML
copetersRG
southcentralus
60582a10-b9fd-49f1-a546-c4194134bba8


### Create sample input and output dataset

In [3]:
from azureml.core import Dataset

datastore = ws.get_default_datastore()
datastore.upload_files(files = ['./features.csv', './labels.csv'],
                       target_path = 'sklearn_regression/',
                       overwrite = True)

input_dataset = Dataset.Tabular.from_delimited_files(path = [(datastore, 'sklearn_regression/features.csv')])
output_dataset = Dataset.Tabular.from_delimited_files(path = [(datastore, 'sklearn_regression/labels.csv')])

Uploading an estimated of 2 files
Uploading ./features.csv
Uploading ./labels.csv
Uploaded ./labels.csv, 1 files out of an estimated total of 2
Uploaded ./features.csv, 2 files out of an estimated total of 2
Uploaded 2 files


### Register Model

You can add tags and descriptions to your Models. Note you need to have a `sklearn_regression_model.pkl` file in the current directory. This file is generated by the 01 notebook. The below call registers that file as a Model with the same name `sklearn_regression_model.pkl` in the workspace.

Using tags, you can track useful information such as the name and version of the machine learning library used to train the model. Note that tags must be alphanumeric.

In [6]:
from azureml.core.model import Model
#from azureml.core.resource_configuration import ResourceConfiguration

model = Model.register(model_path="sklearn_regression_model.pkl",
                       model_name="sklearn_regression_model.pkl",
                       tags={'area': "diabetes", 'type': "regression"},
                       description="Ridge regression model to predict diabetes",
                       workspace=ws)

Registering model sklearn_regression_model.pkl


### Create Environment

You can now create and/or use an Environment object when deploying a Webservice. The Environment can have been previously registered with your Workspace, or it will be registered with it as a part of the Webservice deployment. Only Environments that were created using azureml-defaults version 1.0.48 or later will work with this new handling however.

More information can be found in our [using environments notebook](../training/using-environments/using-environments.ipynb).

In [7]:
from azureml.core import Environment

env = Environment.from_conda_specification(name='deploytocloudenv', file_path='myenv.yml')

# This is optional at this point
# env.register(workspace=ws)

## Create Inference Configuration

There is now support for a source directory, you can upload an entire folder from your local machine as dependencies for the Webservice.
Note: in that case, your entry_script, conda_file, and extra_docker_file_steps paths are relative paths to the source_directory path.

Sample code for using a source directory:

```python
inference_config = InferenceConfig(source_directory="C:/abc",
                                   runtime= "python", 
                                   entry_script="x/y/score.py",
                                   conda_file="env/myenv.yml", 
                                   extra_docker_file_steps="helloworld.txt")
```

 - source_directory = holds source path as string, this entire folder gets added in image so its really easy to access any files within this folder or subfolder
 - runtime = Which runtime to use for the image. Current supported runtimes are 'spark-py' and 'python
 - entry_script = contains logic specific to initializing your model and running predictions
 - conda_file = manages conda and python package dependencies.
 - extra_docker_file_steps = optional: any extra steps you want to inject into docker file

In [16]:
from azureml.core.model import InferenceConfig

inference_config = InferenceConfig(entry_script="score.py", environment=env)

In [22]:
from azureml.core.compute import AksCompute, ComputeTarget

# Use the default configuration (you can also provide parameters to customize this).
# For example, to create a dev/test cluster, use:
# prov_config = AksCompute.provisioning_configuration(cluster_purpose = AksCompute.ClusterPurpose.DEV_TEST)
prov_config = AksCompute.provisioning_configuration()

aks_name = 'myaks11'
# Create the cluster
aks_target = ComputeTarget.create(workspace = ws,
                                    name = aks_name,
                                    provisioning_configuration = prov_config)

# Wait for the create process to complete
aks_target.wait_for_completion(show_output = True)

Creating..........................................................................................................................................................
SucceededProvisioning operation finished, operation "Succeeded"


### Deploy Model as Webservice on Azure Container Instance

Note that the service creation can take few minutes.

In [25]:
from azureml.core.webservice import AksWebservice, Webservice
from azureml.exceptions import WebserviceException

deployment_config = AksWebservice.deploy_configuration(cpu_cores=1, memory_gb=1)
service_name = 'service11'

service = Model.deploy(ws, service_name, [model], inference_config, deployment_config, aks_target)

service.wait_for_deployment(True)
print(service.state)

Running...............
SucceededAKS service creation operation finished, operation "Succeeded"
Healthy


In [26]:
service

AksWebservice(workspace=Workspace.create(name='Azure-ML', subscription_id='60582a10-b9fd-49f1-a546-c4194134bba8', resource_group='copetersRG'), name=service11, image_id=None, compute_type=AKS, state=Healthy, scoring_uri=http://13.66.39.137:80/api/v1/service/service11/score, tags={}, properties={'azureml.git.repository_uri': 'https://github.com/lostmygithubaccount/dkdc.git', 'mlflow.source.git.repoURL': 'https://github.com/lostmygithubaccount/dkdc.git', 'azureml.git.branch': 'master', 'mlflow.source.git.branch': 'master', 'azureml.git.commit': 'c6b43516c75f48fbccf10d17d3fb744a57dcfb68', 'mlflow.source.git.commit': 'c6b43516c75f48fbccf10d17d3fb744a57dcfb68', 'azureml.git.dirty': 'True'})

#### Test web service

In [29]:
import json
test_sample = json.dumps({'data': [
    [1,2,3,4,5,6,7,8,9,10], 
    [10,9,8,7,6,5,4,3,2,1]
]})

test_sample_encoded = bytes(test_sample, encoding='utf8')
prediction = service.run(input_data=test_sample_encoded)
print(prediction)

[5215.1981315798685, 3726.995485938578]


#### Delete ACI to clean up

In [ ]:
service.delete()

### Model Profiling

You can also take advantage of the profiling feature to estimate CPU and memory requirements for models.

```python
profile = Model.profile(ws, "profilename", [model], inference_config, test_sample)
profile.wait_for_profiling(True)
profiling_results = profile.get_results()
print(profiling_results)
```

### Model Packaging

If you want to build a Docker image that encapsulates your model and its dependencies, you can use the model packaging option. The output image will be pushed to your workspace's ACR.

You must include an Environment object in your inference configuration to use `Model.package()`.

```python
package = Model.package(ws, [model], inference_config)
package.wait_for_creation(show_output=True)  # Or show_output=False to hide the Docker build logs.
package.pull()
```

Instead of a fully-built image, you can also generate a Dockerfile and download all the assets needed to build an image on top of your Environment.

```python
package = Model.package(ws, [model], inference_config, generate_dockerfile=True)
package.wait_for_creation(show_output=True)
package.save("./local_context_dir")
```